<a href="https://colab.research.google.com/github/faezesarlakifar/AllerTrans/blob/main/additional-experiments/1D-CNN%20(with%20different%20feature%20vectors).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @markdown import necessaries
from google.colab import drive
from tqdm.notebook import tqdm
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import matthews_corrcoef
from sklearn.metrics import roc_auc_score

In [2]:
# @markdown mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
path = '/content/drive/MyDrive/allergen-detection/embeddings/'

In [4]:
# @markdown load dataset

esm_path = path+'esm-embeddings-with-id/'
esm_train = pd.read_csv(esm_path+'esm_train.csv', index_col=0)
esm_test = pd.read_csv(esm_path+'esm_test.csv', index_col=0)

protbert_path = path+'protBERT-embeddings-with-id/'
protbert_train = pd.read_csv(protbert_path+'protBERT_train.csv', index_col=0)
protbert_test = pd.read_csv(protbert_path+'protBERT_test.csv', index_col=0)

In [ ]:
esm_train.head()

,0,1,2,3,4,5,6,7,8,9,...,1272,1273,1274,1275,1276,1277,1278,1279,Label,id
0,-0.003870,-0.033864,-0.047640,0.042528,0.008789,-0.115301,0.116311,-0.056773,-0.034452,0.007469,...,0.006811,0.046086,-0.028301,0.006903,0.048489,-0.044930,-0.022138,0.003444,0,N_98814
1,-0.027616,-0.047448,-0.092680,-0.037921,-0.064493,-0.085917,0.032686,-0.104749,0.044072,0.136332,...,-0.005730,-0.007972,-0.110330,0.143354,0.079251,-0.200938,-0.064969,-0.010540,1,P_8856
2,-0.060472,-0.009816,-0.085757,-0.042129,-0.005332,-0.114610,-0.023706,-0.015580,0.004648,0.080403,...,-0.015167,-0.000249,-0.056284,0.046674,0.075886,-0.093865,-0.053575,0.069379,1,P_7716
3,-0.001879,-0.132213,0.030315,0.017521,0.093525,-0.132252,0.067915,0.093191,0.031767,0.070513,...,-0.122606,-0.032942,0.069062,-0.176981,0.018370,-0.074525,0.126718,0.107524,0,N_326869
4,-0.004742,-0.012900,-0.069494,0.027048,-0.110435,-0.009641,0.055032,-0.205043,0.036597,0.031489,...,0.036046,-0.113270,-0.020870,-0.023917,0.021013,-0.146265,-0.051615,0.176472,1,P_8031


In [ ]:
esm_test.head()

,0,1,2,3,4,5,6,7,8,9,...,1272,1273,1274,1275,1276,1277,1278,1279,Label,id
0,-0.013756,-0.056206,-0.010463,0.078954,-0.013414,-0.036181,0.101913,-0.078302,-0.029859,0.155479,...,-0.108262,-0.018523,0.023435,-0.086668,0.084220,-0.138123,0.000659,0.111366,0,N_528450
1,0.138220,-0.015468,-0.014704,-0.045518,-0.017561,0.011617,-0.093774,-0.057526,-0.016540,0.202359,...,-0.109726,0.094971,0.022741,-0.153795,0.289212,-0.096826,-0.012760,-0.088842,0,N_479266
2,0.008007,-0.054583,-0.012078,0.037711,-0.046313,-0.094285,0.004338,-0.064758,-0.096635,0.059523,...,-0.036893,0.058147,-0.039131,0.076175,0.136362,-0.062218,-0.023998,0.022915,1,P_271
3,-0.074227,-0.024113,-0.001488,-0.026625,-0.056001,0.005185,-0.002422,-0.120850,-0.008313,-0.003280,...,-0.065336,0.041332,-0.066702,0.075858,-0.057383,-0.165823,0.100414,0.105647,0,N_217429
4,0.035093,-0.027917,0.017686,0.011974,0.123146,-0.051364,-0.024959,0.176133,-0.043006,-0.027118,...,-0.204552,-0.052534,-0.056887,0.034603,0.107830,-0.013466,0.075048,-0.013248,1,P_7456


In [ ]:
protbert_train.head()

,0,1,2,3,4,5,6,7,8,9,...,1016,1017,1018,1019,1020,1021,1022,1023,Label,id
0,0.02951,-0.038240,0.042880,-0.03029,-0.01486,0.09100,-0.02155,-0.07270,0.08460,-0.00852,...,0.017780,-0.064150,0.036220,-0.07806,-0.014550,-0.009850,0.03534,0.02070,1,P_194
1,0.02860,-0.078300,0.044460,-0.03784,0.01002,0.10065,-0.03247,-0.06560,0.07630,-0.01620,...,0.009390,-0.072000,0.019970,-0.10583,-0.012480,-0.009700,0.04852,0.02205,1,P_6357
2,-0.04730,-0.002228,-0.031100,-0.01897,0.02316,0.01805,0.02126,-0.07916,-0.03537,-0.05865,...,-0.019170,0.001675,0.035900,0.00710,0.021940,-0.014656,0.03111,-0.01790,0,N_341543
3,0.01968,0.079900,0.001131,0.03240,0.02434,-0.07947,-0.01135,-0.12840,-0.03787,0.02830,...,-0.000449,-0.206700,-0.005222,0.01127,0.012085,0.014510,0.04248,-0.02217,0,N_257764
4,0.00896,0.022220,0.062230,0.00716,0.03140,0.03087,-0.04764,-0.00948,0.04837,0.02878,...,-0.037500,-0.075500,-0.021120,0.04892,-0.012870,0.023820,-0.04000,-0.06240,0,N_243343


In [ ]:
protbert_test.head()

,0,1,2,3,4,5,6,7,8,9,...,1016,1017,1018,1019,1020,1021,1022,1023,Label,id
0,-0.01852,0.08295,0.03818,-0.01380,-0.015580,-0.020190,0.04840,-0.03552,-0.000656,-0.060150,...,-0.028060,0.01277,0.047550,-0.03232,-0.02292,0.052200,0.02275,-0.023390,1,P_2249
1,-0.02400,0.00681,0.03004,-0.01816,-0.046100,-0.064500,-0.03568,-0.14220,0.004387,-0.063600,...,-0.007454,-0.04565,0.000140,-0.07220,-0.03534,0.021550,0.07184,0.006330,1,P_8311
2,-0.01677,-0.04846,0.03910,0.00860,-0.052100,-0.074800,-0.00942,-0.06480,-0.004760,-0.021060,...,0.003250,-0.04965,-0.008705,-0.03333,-0.04065,-0.021400,0.04794,0.054840,1,P_9307
3,0.01593,-0.05563,0.02385,-0.04205,-0.000392,-0.002066,-0.01698,-0.18870,0.046720,0.049530,...,0.042270,-0.12450,0.048220,0.06020,0.05258,0.003897,-0.00058,-0.027620,0,N_238995
4,-0.00402,-0.08496,0.06910,0.04944,0.025000,-0.015114,-0.06665,-0.06140,-0.004950,0.002924,...,-0.007980,-0.02780,0.069300,0.03119,-0.00836,-0.053560,0.05910,0.003437,0,N_255457


In [5]:
# @title Concatenate training DataFrames
train = pd.merge(
    left=esm_train,
    right=protbert_train,
    on='id'
)
train.head()

,0_x,1_x,2_x,3_x,4_x,5_x,6_x,7_x,8_x,9_x,...,1015_y,1016_y,1017_y,1018_y,1019_y,1020_y,1021_y,1022_y,1023_y,Label_y
0,-0.003870,-0.033864,-0.047640,0.042528,0.008789,-0.115301,0.116311,-0.056773,-0.034452,0.007469,...,-0.058780,-0.01512,0.01888,0.02798,0.005867,-0.03363,-0.04044,-0.02450,0.034730,0
1,-0.027616,-0.047448,-0.092680,-0.037921,-0.064493,-0.085917,0.032686,-0.104749,0.044072,0.136332,...,0.034600,0.01502,-0.07150,0.05087,-0.082300,-0.01258,-0.01215,0.02357,0.011284,1
2,-0.060472,-0.009816,-0.085757,-0.042129,-0.005332,-0.114610,-0.023706,-0.015580,0.004648,0.080403,...,-0.010414,0.01288,-0.05704,0.01997,0.014015,-0.02370,0.06256,0.04718,0.013390,1
3,-0.001879,-0.132213,0.030315,0.017521,0.093525,-0.132252,0.067915,0.093191,0.031767,0.070513,...,-0.043850,-0.02454,-0.02739,0.04074,0.035600,-0.04605,-0.02040,0.04890,-0.019760,0
4,-0.004742,-0.012900,-0.069494,0.027048,-0.110435,-0.009641,0.055032,-0.205043,0.036597,0.031489,...,0.000988,-0.00882,-0.02419,0.05435,-0.088200,-0.04056,-0.06082,0.02296,0.020160,1


In [6]:
train['Label'] = train['Label_x']
train = train.drop(columns=['Label_x', 'Label_y'])
train.head()

,0_x,1_x,2_x,3_x,4_x,5_x,6_x,7_x,8_x,9_x,...,1015_y,1016_y,1017_y,1018_y,1019_y,1020_y,1021_y,1022_y,1023_y,Label
0,-0.003870,-0.033864,-0.047640,0.042528,0.008789,-0.115301,0.116311,-0.056773,-0.034452,0.007469,...,-0.058780,-0.01512,0.01888,0.02798,0.005867,-0.03363,-0.04044,-0.02450,0.034730,0
1,-0.027616,-0.047448,-0.092680,-0.037921,-0.064493,-0.085917,0.032686,-0.104749,0.044072,0.136332,...,0.034600,0.01502,-0.07150,0.05087,-0.082300,-0.01258,-0.01215,0.02357,0.011284,1
2,-0.060472,-0.009816,-0.085757,-0.042129,-0.005332,-0.114610,-0.023706,-0.015580,0.004648,0.080403,...,-0.010414,0.01288,-0.05704,0.01997,0.014015,-0.02370,0.06256,0.04718,0.013390,1
3,-0.001879,-0.132213,0.030315,0.017521,0.093525,-0.132252,0.067915,0.093191,0.031767,0.070513,...,-0.043850,-0.02454,-0.02739,0.04074,0.035600,-0.04605,-0.02040,0.04890,-0.019760,0
4,-0.004742,-0.012900,-0.069494,0.027048,-0.110435,-0.009641,0.055032,-0.205043,0.036597,0.031489,...,0.000988,-0.00882,-0.02419,0.05435,-0.088200,-0.04056,-0.06082,0.02296,0.020160,1


In [7]:
# @title Concatenate test DataFrames
test = pd.merge(
    left=esm_test,
    right=protbert_test,
    on='id'
)
test.head()

,0_x,1_x,2_x,3_x,4_x,5_x,6_x,7_x,8_x,9_x,...,1015_y,1016_y,1017_y,1018_y,1019_y,1020_y,1021_y,1022_y,1023_y,Label_y
0,-0.013756,-0.056206,-0.010463,0.078954,-0.013414,-0.036181,0.101913,-0.078302,-0.029859,0.155479,...,-0.002998,0.026950,-0.05620,0.06690,0.02548,-0.006428,0.04312,0.06300,0.008500,0
1,0.138220,-0.015468,-0.014704,-0.045518,-0.017561,0.011617,-0.093774,-0.057526,-0.016540,0.202359,...,0.035600,-0.014534,-0.08590,0.03397,0.03726,0.012790,-0.02563,0.09100,0.027910,0
2,0.008007,-0.054583,-0.012078,0.037711,-0.046313,-0.094285,0.004338,-0.064758,-0.096635,0.059523,...,-0.038570,-0.009390,-0.03925,0.04712,0.03078,0.004520,-0.02391,0.06647,-0.003840,1
3,-0.074227,-0.024113,-0.001488,-0.026625,-0.056001,0.005185,-0.002422,-0.120850,-0.008313,-0.003280,...,-0.049500,-0.029420,-0.08580,-0.02002,0.03270,-0.029040,-0.06335,0.01161,-0.006042,0
4,0.035093,-0.027917,0.017686,0.011974,0.123146,-0.051364,-0.024959,0.176133,-0.043006,-0.027118,...,0.029170,-0.052730,-0.06920,0.03854,0.11456,-0.035220,-0.01343,0.10626,0.055600,1


In [8]:
test['Label'] = test['Label_x']
test = test.drop(columns=['Label_x', 'Label_y'])
test.head()

,0_x,1_x,2_x,3_x,4_x,5_x,6_x,7_x,8_x,9_x,...,1015_y,1016_y,1017_y,1018_y,1019_y,1020_y,1021_y,1022_y,1023_y,Label
0,-0.013756,-0.056206,-0.010463,0.078954,-0.013414,-0.036181,0.101913,-0.078302,-0.029859,0.155479,...,-0.002998,0.026950,-0.05620,0.06690,0.02548,-0.006428,0.04312,0.06300,0.008500,0
1,0.138220,-0.015468,-0.014704,-0.045518,-0.017561,0.011617,-0.093774,-0.057526,-0.016540,0.202359,...,0.035600,-0.014534,-0.08590,0.03397,0.03726,0.012790,-0.02563,0.09100,0.027910,0
2,0.008007,-0.054583,-0.012078,0.037711,-0.046313,-0.094285,0.004338,-0.064758,-0.096635,0.059523,...,-0.038570,-0.009390,-0.03925,0.04712,0.03078,0.004520,-0.02391,0.06647,-0.003840,1
3,-0.074227,-0.024113,-0.001488,-0.026625,-0.056001,0.005185,-0.002422,-0.120850,-0.008313,-0.003280,...,-0.049500,-0.029420,-0.08580,-0.02002,0.03270,-0.029040,-0.06335,0.01161,-0.006042,0
4,0.035093,-0.027917,0.017686,0.011974,0.123146,-0.051364,-0.024959,0.176133,-0.043006,-0.027118,...,0.029170,-0.052730,-0.06920,0.03854,0.11456,-0.035220,-0.01343,0.10626,0.055600,1


In [9]:
df_train = train
df_test = test

In [10]:
concat_train = train
concat_test = test

# Preprocess and Prepare Dataset

In [11]:
df_train = df_train.drop('id', axis=1)
df_test = df_test.drop('id', axis=1)

In [12]:
# Convert Label to integer
df_train['Label'] = df_train['Label'].astype(int)
df_test['Label'] = df_test['Label'].astype(int)

# Split data into train and validation sets
X_train = df_train.drop('Label', axis=1)
y_train = df_train['Label']
X_test = df_test.drop('Label', axis=1)
y_test = df_test['Label']


In [13]:
# Create new feature names
feature_names = ['feat_'+str(i) for i in range(X_train.shape[1])]

# Rename columns
X_train.columns = feature_names
X_test.columns = feature_names

In [14]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_scaled = torch.from_numpy(X_train_scaled).float()
X_test_scaled = torch.from_numpy(X_test_scaled).float()

X_train = X_train_scaled
X_test = X_test_scaled

In [15]:
# Convert to numpy arrays
y_train = np.array(y_train)
y_test = np.array(y_test)

# Create Tensors from Labels
y_train = torch.from_numpy(y_train).long()
y_test = torch.from_numpy(y_test).long()

In [16]:
X_train.shape

torch.Size([16120, 2304])

In [17]:
y_train.shape

torch.Size([16120])

In [18]:
X_train_cat = X_train
y_train_cat = y_train

X_test_cat = X_test
y_test_cat = y_test

# ProtT5

In [19]:
# @markdown Load Dataset
path = '/content/drive/MyDrive/allergen-detection/embeddings/'

protbert_path = path+'protBERT-embeddings-with-id/'
protbert_train = pd.read_csv(protbert_path+'protBERT_train.csv', index_col=0)
protbert_test = pd.read_csv(protbert_path+'protBERT_test.csv', index_col=0)


df_train = protbert_train
df_test = protbert_test

df_train = df_train.drop('id', axis=1)
df_test = df_test.drop('id', axis=1)

# Convert Label to integer
df_train['Label'] = df_train['Label'].astype(int)
df_test['Label'] = df_test['Label'].astype(int)

# Split data into train and validation sets
X_train = df_train.drop('Label', axis=1)
y_train = df_train['Label']
X_test = df_test.drop('Label', axis=1)
y_test = df_test['Label']

# Create new feature names
feature_names = ['feat_'+str(i) for i in range(X_train.shape[1])]

# Rename columns
X_train.columns = feature_names
X_test.columns = feature_names

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_scaled = torch.from_numpy(X_train_scaled).float()
X_test_scaled = torch.from_numpy(X_test_scaled).float()

X_train = X_train_scaled
X_test = X_test_scaled

# Convert to numpy arrays
y_train = np.array(y_train)
y_test = np.array(y_test)

# Create Tensors from Labels
y_train = torch.from_numpy(y_train).long()
y_test = torch.from_numpy(y_test).long()

## 1D-CNN as Classifier

In [20]:
from torch.utils.data import TensorDataset

In [21]:
# ConvNet Model
class ConvNet(nn.Module):
    def __init__(self, in_channels=1):
      super().__init__()

      # self.conv1 = nn.Conv1d(in_channels, 32, 3)
      # self.conv2 = nn.Conv1d(32, 64, 3)

      self.conv1 = nn.Conv1d(in_channels, 32, 3)
      self.conv2 = nn.Conv1d(32, 64, 3)

      self.conv_dropout = nn.Dropout(p=0.2)

      self.fc1 = nn.Linear(1020 * 32, 128)
      self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool1d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [23]:
model = ConvNet(1)
model.to(device)
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters())
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Batch size
batch_size = 64

In [ ]:
# Convert to numpy arrays
y_train = np.array(y_train)
y_test = np.array(y_test)

# Create Tensors from Labels
y_train = torch.from_numpy(y_train).long()
y_test = torch.from_numpy(y_test).long()

In [24]:
# Define TensorDataset
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64)


test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=64)


### Train & Evaluate CNN model

In [25]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, matthews_corrcoef

# Training loop
epochs = 10
for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        inputs, labels = batch
        inputs = inputs.unsqueeze(1).to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# --- Evaluation phase ---
model.eval()
all_preds = []
all_labels = []
all_scores = []

with torch.no_grad():
    for batch in test_loader:
        inputs, labels = batch
        inputs = inputs.unsqueeze(1).to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        # Predictions
        probs = F.softmax(outputs, dim=1)
        _, predicted = torch.max(probs.data, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_scores.extend(probs[:,1].cpu().numpy())  # Probability of positive class

# Convert to NumPy arrays
y_pred = np.array(all_preds)
y_true = np.array(all_labels)
y_scores = np.array(all_scores)

# Metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
_, recall_per_class, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
roc_auc = roc_auc_score(y_true, y_scores)
mcc = matthews_corrcoef(y_true, y_pred)

# Print metrics
print(f"\n--- Evaluation Metrics ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Negative Class Recall: {recall_per_class[0]:.4f}")
print(f"Positive Class Recall: {recall_per_class[1]:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"MCC: {mcc:.4f}")

Epoch 1, Loss: 0.0908
Epoch 2, Loss: 0.0426
Epoch 3, Loss: 0.0346
Epoch 4, Loss: 0.0231
Epoch 5, Loss: 0.0229
Epoch 6, Loss: 0.0140
Epoch 7, Loss: 0.0139
Epoch 8, Loss: 0.0249
Epoch 9, Loss: 0.0191
Epoch 10, Loss: 0.0331

--- Evaluation Metrics ---
Accuracy: 0.8794
Precision: 0.8868
Recall: 0.8794
F1 Score: 0.8788
Negative Class Recall: 0.9484
Positive Class Recall: 0.8104
ROC AUC: 0.9607
MCC: 0.7661


# Prot-ESM

In [26]:
X_train = X_train_cat
y_train = y_train_cat

X_test = X_test_cat
y_test = y_test_cat

## 1D-CNN as Classifier

In [ ]:
# ConvNet Model
class ConvNet(nn.Module):
    def __init__(self, in_channels=1):
      super().__init__()

      # self.conv1 = nn.Conv1d(in_channels, 32, 3)
      # self.conv2 = nn.Conv1d(32, 64, 3)

      self.conv1 = nn.Conv1d(in_channels, 32, 3)
      self.conv2 = nn.Conv1d(32, 64, 3)

      self.conv_dropout = nn.Dropout(p=0.2)

      self.fc1 = nn.Linear(2300 * 32, 128)
      self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool1d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
model = ConvNet(1)
model.to(device)
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters())
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-5)

# Batch size
batch_size = 64

In [ ]:
# Convert to numpy arrays
y_train = np.array(y_train)
y_test = np.array(y_test)

# Create Tensors from Labels
y_train = torch.from_numpy(y_train).long()
y_test = torch.from_numpy(y_test).long()

In [ ]:
# Define TensorDataset
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64)


test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=64)

### Train & Evaluate CNN model

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, matthews_corrcoef

# Training loop
epochs = 10
for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        inputs, labels = batch
        inputs = inputs.unsqueeze(1).to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# --- Evaluation phase ---
model.eval()
all_preds = []
all_labels = []
all_scores = []

with torch.no_grad():
    for batch in test_loader:
        inputs, labels = batch
        inputs = inputs.unsqueeze(1).to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        # Predictions
        probs = F.softmax(outputs, dim=1)
        _, predicted = torch.max(probs.data, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_scores.extend(probs[:,1].cpu().numpy())  # Probability of positive class

# Convert to NumPy arrays
y_pred = np.array(all_preds)
y_true = np.array(all_labels)
y_scores = np.array(all_scores)

# Metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
_, recall_per_class, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
roc_auc = roc_auc_score(y_true, y_scores)
mcc = matthews_corrcoef(y_true, y_pred)

# Print metrics
print(f"\n--- Evaluation Metrics ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Negative Class Recall: {recall_per_class[0]:.4f}")
print(f"Positive Class Recall: {recall_per_class[1]:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"MCC: {mcc:.4f}")

Epoch 1, Loss: 0.3442
Epoch 2, Loss: 0.2817
Epoch 3, Loss: 0.2262
Epoch 4, Loss: 0.2071
Epoch 5, Loss: 0.1964
Epoch 6, Loss: 0.1885
Epoch 7, Loss: 0.1815
Epoch 8, Loss: 0.1747
Epoch 9, Loss: 0.1673
Epoch 10, Loss: 0.1596

--- Evaluation Metrics ---
Accuracy: 0.9186
Precision: 0.9267
Recall: 0.9186
F1 Score: 0.9182
Negative Class Recall: 0.9876
Positive Class Recall: 0.8496
ROC AUC: 0.9795
MCC: 0.8453


In [ ]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, matthews_corrcoef

# Training loop
epochs = 10
for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        inputs, labels = batch
        inputs = inputs.unsqueeze(1).to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# --- Evaluation phase ---
model.eval()
all_preds = []
all_labels = []
all_scores = []

with torch.no_grad():
    for batch in test_loader:
        inputs, labels = batch
        inputs = inputs.unsqueeze(1).to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        # Predictions
        probs = F.softmax(outputs, dim=1)
        _, predicted = torch.max(probs.data, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_scores.extend(probs[:,1].cpu().numpy())  # Probability of positive class

# Convert to NumPy arrays
y_pred = np.array(all_preds)
y_true = np.array(all_labels)
y_scores = np.array(all_scores)

# Metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
_, recall_per_class, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
roc_auc = roc_auc_score(y_true, y_scores)
mcc = matthews_corrcoef(y_true, y_pred)

# Print metrics
print(f"\n--- Evaluation Metrics ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Negative Class Recall: {recall_per_class[0]:.4f}")
print(f"Positive Class Recall: {recall_per_class[1]:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"MCC: {mcc:.4f}")

Epoch 1, Loss: 0.1880
Epoch 2, Loss: 0.1488
Epoch 3, Loss: 0.1194
Epoch 4, Loss: 0.0961
Epoch 5, Loss: 0.1192
Epoch 6, Loss: 0.0993
Epoch 7, Loss: 0.1019
Epoch 8, Loss: 0.0140
Epoch 9, Loss: 0.0095
Epoch 10, Loss: 0.0365

--- Evaluation Metrics ---
Accuracy: 0.8253
Precision: 0.8668
Recall: 0.8253
F1 Score: 0.8202
Negative Class Recall: 0.9935
Positive Class Recall: 0.6571
ROC AUC: 0.9615
MCC: 0.6909


# ProtAAC

### Load & Preprocess Data

In [27]:
path = '/content/drive/MyDrive/allergen-detection/embeddings/'

In [28]:
# @title load dataset

AAC_path = '/content/drive/MyDrive/allergen-detection/ACC/'
AAC_train = pd.read_csv(AAC_path+'df_train_ACC.csv')
AAC_test = pd.read_csv(AAC_path+'df_valid_ACC.csv')

protbert_path = path+'protBERT-embeddings-with-id/'
protbert_train = pd.read_csv(protbert_path+'protBERT_train.csv', index_col=0)
protbert_test = pd.read_csv(protbert_path+'protBERT_test.csv', index_col=0)

In [29]:
# @title Concatenate training DataFrames
train = pd.merge(
    left=AAC_train,
    right=protbert_train,
    on='id'
)
train.head()

,AC111,AC112,AC113,AC114,AC115,AC221,AC222,AC223,AC224,AC225,...,1015,1016,1017,1018,1019,1020,1021,1022,1023,Label_y
0,-0.001588,-0.010216,0.009495,0.010611,-0.007591,0.001921,0.004524,0.002681,0.003329,0.003225,...,0.028340,-0.01420,-0.04608,0.01439,0.04850,-0.014435,-0.053040,0.039030,0.03876,1
1,0.002542,-0.000007,-0.003138,0.006212,-0.000897,0.006240,0.000340,0.007036,-0.001322,-0.003758,...,-0.009865,0.01421,-0.13330,-0.02829,0.12120,0.013130,0.092700,0.002281,-0.01819,0
2,0.003587,-0.004062,-0.007729,-0.005593,-0.003180,0.002744,-0.004103,0.004514,-0.000169,0.000349,...,-0.002352,-0.00444,-0.04257,0.04500,-0.08440,-0.033630,-0.055150,0.029940,0.03128,1
3,-0.000019,0.001728,0.004911,-0.003602,-0.001826,-0.000710,-0.000638,-0.005235,0.004258,-0.005560,...,-0.000987,0.02390,-0.08080,0.01773,-0.01240,0.036250,0.014015,0.054900,0.03635,1
4,0.005278,0.001208,0.008769,0.007772,0.001478,-0.005907,0.006403,0.005930,0.007527,0.006052,...,0.016880,0.02052,-0.05603,0.00396,0.01179,-0.034150,-0.037750,-0.054960,-0.00402,0


In [30]:
train['Label'] = train['Label_x']
train = train.drop(columns=['Label_x', 'Label_y', 'id'])
train.head()

,AC111,AC112,AC113,AC114,AC115,AC221,AC222,AC223,AC224,AC225,...,1015,1016,1017,1018,1019,1020,1021,1022,1023,Label
0,-0.001588,-0.010216,0.009495,0.010611,-0.007591,0.001921,0.004524,0.002681,0.003329,0.003225,...,0.028340,-0.01420,-0.04608,0.01439,0.04850,-0.014435,-0.053040,0.039030,0.03876,1
1,0.002542,-0.000007,-0.003138,0.006212,-0.000897,0.006240,0.000340,0.007036,-0.001322,-0.003758,...,-0.009865,0.01421,-0.13330,-0.02829,0.12120,0.013130,0.092700,0.002281,-0.01819,0
2,0.003587,-0.004062,-0.007729,-0.005593,-0.003180,0.002744,-0.004103,0.004514,-0.000169,0.000349,...,-0.002352,-0.00444,-0.04257,0.04500,-0.08440,-0.033630,-0.055150,0.029940,0.03128,1
3,-0.000019,0.001728,0.004911,-0.003602,-0.001826,-0.000710,-0.000638,-0.005235,0.004258,-0.005560,...,-0.000987,0.02390,-0.08080,0.01773,-0.01240,0.036250,0.014015,0.054900,0.03635,1
4,0.005278,0.001208,0.008769,0.007772,0.001478,-0.005907,0.006403,0.005930,0.007527,0.006052,...,0.016880,0.02052,-0.05603,0.00396,0.01179,-0.034150,-0.037750,-0.054960,-0.00402,0


In [31]:
# @title Concatenate test DataFrames
test = pd.merge(
    left=AAC_test,
    right=protbert_test,
    on='id'
)
test.head()

,AC111,AC112,AC113,AC114,AC115,AC221,AC222,AC223,AC224,AC225,...,1015,1016,1017,1018,1019,1020,1021,1022,1023,Label_y
0,-0.002709,-0.003017,0.003407,-0.002346,-0.000988,-0.001712,0.000459,-0.000865,0.001448,-0.001776,...,-0.019960,0.000692,-0.04890,0.06192,-0.021940,0.003428,0.03090,0.057830,-0.005928,1
1,-0.001621,-0.003559,0.000425,-0.003073,-0.004361,0.002970,-0.000206,-0.003230,0.001556,-0.000295,...,-0.010345,-0.003754,-0.10200,0.02716,0.036960,-0.006160,-0.05157,-0.005333,0.026460,0
2,0.009525,0.011570,0.006164,0.004729,-0.002226,0.004134,-0.006477,0.004354,0.006389,-0.002751,...,-0.038670,0.029630,-0.08563,0.09766,0.008064,-0.037600,0.02690,0.059500,0.028000,1
3,-0.000123,-0.000442,0.003143,0.002167,0.000882,0.008620,0.003198,0.004305,0.006099,0.006787,...,-0.032600,-0.050960,-0.02983,0.05360,0.001793,-0.046050,0.02757,0.039180,0.024460,0
4,-0.001543,-0.007223,-0.006352,-0.006966,0.002633,0.002969,0.004972,0.003986,0.004642,0.001234,...,-0.046420,-0.038360,-0.05286,0.04060,0.009840,0.000133,0.02797,0.119500,0.075800,1


In [32]:
test['Label'] = test['Label_x']
test = test.drop(columns=['Label_x', 'Label_y', 'id'])
test.head()

,AC111,AC112,AC113,AC114,AC115,AC221,AC222,AC223,AC224,AC225,...,1015,1016,1017,1018,1019,1020,1021,1022,1023,Label
0,-0.002709,-0.003017,0.003407,-0.002346,-0.000988,-0.001712,0.000459,-0.000865,0.001448,-0.001776,...,-0.019960,0.000692,-0.04890,0.06192,-0.021940,0.003428,0.03090,0.057830,-0.005928,1
1,-0.001621,-0.003559,0.000425,-0.003073,-0.004361,0.002970,-0.000206,-0.003230,0.001556,-0.000295,...,-0.010345,-0.003754,-0.10200,0.02716,0.036960,-0.006160,-0.05157,-0.005333,0.026460,0
2,0.009525,0.011570,0.006164,0.004729,-0.002226,0.004134,-0.006477,0.004354,0.006389,-0.002751,...,-0.038670,0.029630,-0.08563,0.09766,0.008064,-0.037600,0.02690,0.059500,0.028000,1
3,-0.000123,-0.000442,0.003143,0.002167,0.000882,0.008620,0.003198,0.004305,0.006099,0.006787,...,-0.032600,-0.050960,-0.02983,0.05360,0.001793,-0.046050,0.02757,0.039180,0.024460,0
4,-0.001543,-0.007223,-0.006352,-0.006966,0.002633,0.002969,0.004972,0.003986,0.004642,0.001234,...,-0.046420,-0.038360,-0.05286,0.04060,0.009840,0.000133,0.02797,0.119500,0.075800,1


#### Data Preprocessing

In [33]:
df_train = train
df_test = test

In [34]:
# Convert Label to integer
df_train['Label'] = df_train['Label'].astype(int)
df_test['Label'] = df_test['Label'].astype(int)

# Split data into train and validation sets
X_train = df_train.drop('Label', axis=1)
y_train = df_train['Label']
X_test = df_test.drop('Label', axis=1)
y_test = df_test['Label']

In [35]:
# Create new feature names
feature_names = ['feat_'+str(i) for i in range(X_train.shape[1])]

# Rename columns
X_train.columns = feature_names
X_test.columns = feature_names

In [36]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_scaled = torch.from_numpy(X_train_scaled).float()
X_test_scaled = torch.from_numpy(X_test_scaled).float()

X_train = X_train_scaled
X_test = X_test_scaled

In [37]:
# Convert to numpy arrays
y_train = np.array(y_train)
y_test = np.array(y_test)

# Create Tensors from Labels
y_train = torch.from_numpy(y_train).long()
y_test = torch.from_numpy(y_test).long()

## 1D-CNN as Classifier

In [38]:
from torch.utils.data import TensorDataset

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [40]:
# ConvNet Model
class ConvNet(nn.Module):
    def __init__(self, in_channels=1):
      super().__init__()

      # self.conv1 = nn.Conv1d(in_channels, 32, 3)
      # self.conv2 = nn.Conv1d(32, 64, 3)

      self.conv1 = nn.Conv1d(in_channels, 32, 3)
      self.conv2 = nn.Conv1d(32, 64, 3)

      self.conv_dropout = nn.Dropout(p=0.2)

      self.fc1 = nn.Linear(1144 * 32, 128)
      self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool1d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [41]:
model = ConvNet(1)
model.to(device)
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Batch size
batch_size = 64

In [42]:
# Convert to numpy arrays
y_train = np.array(y_train)
y_test = np.array(y_test)

# Create Tensors from Labels
y_train = torch.from_numpy(y_train).long()
y_test = torch.from_numpy(y_test).long()

In [43]:
# Define TensorDataset
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64)


test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=64)

### Train & Evaluate CNN model

In [44]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, matthews_corrcoef

# Training loop
epochs = 10
for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        inputs, labels = batch
        inputs = inputs.unsqueeze(1).to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# --- Evaluation phase ---
model.eval()
all_preds = []
all_labels = []
all_scores = []

with torch.no_grad():
    for batch in test_loader:
        inputs, labels = batch
        inputs = inputs.unsqueeze(1).to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        # Predictions
        probs = F.softmax(outputs, dim=1)
        _, predicted = torch.max(probs.data, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_scores.extend(probs[:,1].cpu().numpy())  # Probability of positive class

# Convert to NumPy arrays
y_pred = np.array(all_preds)
y_true = np.array(all_labels)
y_scores = np.array(all_scores)

# Metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
_, recall_per_class, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
roc_auc = roc_auc_score(y_true, y_scores)
mcc = matthews_corrcoef(y_true, y_pred)

# Print metrics
print(f"\n--- Evaluation Metrics ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Negative Class Recall: {recall_per_class[0]:.4f}")
print(f"Positive Class Recall: {recall_per_class[1]:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"MCC: {mcc:.4f}")

Epoch 1, Loss: 0.0926
Epoch 2, Loss: 0.0931
Epoch 3, Loss: 0.0907
Epoch 4, Loss: 0.0887
Epoch 5, Loss: 0.0890
Epoch 6, Loss: 0.0891
Epoch 7, Loss: 0.0939
Epoch 8, Loss: 0.0906
Epoch 9, Loss: 0.0837
Epoch 10, Loss: 0.0302

--- Evaluation Metrics ---
Accuracy: 0.8948
Precision: 0.9074
Recall: 0.8948
F1 Score: 0.8940
Negative Class Recall: 0.9826
Positive Class Recall: 0.8069
ROC AUC: 0.9707
MCC: 0.8021
